In [1]:
import pandas as pd  
from pandas import Series, DataFrame 
import uproot 
from scipy import stats
from scipy.optimize import curve_fit
from scipy.special import comb
from scipy.stats import chi2
from scipy.special import comb
from scipy.optimize import lsq_linear
import sys
from plot_tools import *
from customStats import *
#import tools
import common_tools
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
# from selection_cuts import selection_nominal
import mplhep as hep
from sklearn.model_selection import train_test_split
plt.style.use(hep.style.CMS)
plt.rcParams['figure.figsize'] = [10,8]
plt.rcParams['font.size'] = 24
plt.figure()
plt.close()
plt.rcParams.update({'figure.figsize':[10,8]})
plt.rcParams.update({'font.size':24})
import tensorflow as tf
import math
import zfit
from zfit import z
import xgboost as xgb
from scipy.interpolate import make_interp_spline
# from loadCutXGB import load_and_cutXGBclfs
from scipy.special import comb
from scipy.optimize import lsq_linear
zfit.settings.set_verbosity(0)
import json
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Oculta los mensajes de INFO y WARNING
from PDFs import *
from utils_efficiency import *
from utils_fits import *

2026-03-18 15:37:15.984187: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 15:37:16.010291: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-18 15:37:16.435334: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/ghcp/miniconda3/envs/haza_wokr_env/lib/python3.8/site-packages/zfit/__init__.py:63: UserWarning: TensorFlow warnings are by default suppressed by zfit. In order to show them, set the environment variable ZFIT_DISABLE_TF_WARNINGS=0. In order to suppress th

#  DATA

In [2]:
import uproot
import pandas as pd

# --- RUTAS DE ARCHIVOS ---
f_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged.root"
f_gen_filtered = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged_Filtered.root"
f_reco_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/RecoGenV2_Angular_Merged.root"  
x_gboost_cut = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/BdtoK0smumu-20251110T171511Z-1-001/MyReweiting/ResultsB0_2022/AntiRadVeto_MC_NoRes_2022_Era1_v0_XGBoost_fom_cut_BDT.root"

vars_gen_to_load = ["gen_cosThetaK", "gen_cosThetaL", "gen_phi", "q2Gen"]
vars_reco_to_load = ["CosThetaK_best", "CosThetaL_best", "Phi_best", "massJ"] 
vars_xgboost_to_load = ["CosThetaK", "CosThetaL", "Phi", "massB_test", "massJ", "TotalWeight"] 

# --- CARGA DE DATOS ---
#Gen NO filt
genNFtr = uproot.open(f_gen)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"1. Gen Non-Filtered (genNFtr) cargado: {len(genNFtr)} eventos")
# Gen Filtered
genFtr = uproot.open(f_gen_filtered)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"2. Gen Filtered (genFtr) cargado: {len(genFtr)} eventos")
# Reco Gen Level
recoGen = uproot.open(f_reco_gen)['ntuple'].arrays(vars_reco_to_load, library='pd')
print(f"3. Reco Gen Level Denom (recoGen) cargado: {len(recoGen)} eventos")
# Final selection 
recoFtr = uproot.open(x_gboost_cut)['treeBd'].arrays(vars_xgboost_to_load, library='pd')
print(f"4. Reco Final (recoFtr) cargado: {len(recoFtr)} eventos")



1. Gen Non-Filtered (genNFtr) cargado: 11589148 eventos
2. Gen Filtered (genFtr) cargado: 307688 eventos
3. Reco Gen Level Denom (recoGen) cargado: 6298017 eventos
4. Reco Final (recoFtr) cargado: 900424 eventos


In [3]:

recoFtr["q2"] = recoFtr["massJ"]**2 
recoGen["q2Gen"] = recoGen["massJ"]**2  

GenNFlt = genNFtr.copy()     
GenFlt  = genFtr.copy()       

RecoGenFlt = recoGen.copy()             
mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
Reco = recoFtr[mask_mass].copy()
#2
eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.03, random_state=549)
eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.1, random_state=22)
eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.1, random_state=22)
eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=22)

a1 = np.array(obs_Gen["gen_cosThetaL"])
a2 = np.array(obs_Gen["gen_cosThetaK"])
a3 = np.array(obs_Gen["gen_phi"])

angles = np.array([a1, a2, a3])
valid_observations_mask = ~np.isnan(angles).any(axis=0)
filtered_data = angles[:, valid_observations_mask]

In [4]:
print("Numero de eventes GEN",len(obs_Gen))
print("Numero de eventes GEN Filtered",len(obs_GenFtr))
print("Numero de eventes Reco Gen filtered",len(obs_RecoGenFtr))
print("Numero de eventes Reco Filtered",len(obs_RecoFtr))


Numero de eventes GEN 347675
Numero de eventes GEN Filtered 30769
Numero de eventes Reco Gen filtered 629802
Numero de eventes Reco Filtered 90043


# COMPUTE EFFICIENSY MODELS

In [5]:
nbx=20
nby=20
nx_gen=4 
ny_gen=4 
nx_rec=4 
ny_rec=4
ylim_ck_a=(0.0, 0.040)
ylim_cl_a=(0.0, 0.040)
ylim_ck_r=(0.0, 0.30)
ylim_cl_r=(0.2, 0.450)
ylim_ck_t=(0.0, 0.008)
ylim_cl_t=(0.0, 0.008)
ylim_p_a=(0.0, 0.06)
ylim_p_r=(0.0, 0.25)
ylim_p_t=(0.0, 0.006)

bin_configs = {
    "bin1":  {"q2_range": [1.1, 2.0],   "nbx": 12, "nby": 12, "nx_gen": 5, "ny_gen": 5, "nx_rec": 5, "ny_rec": 5},
    "bin2":  {"q2_range": [2.0, 4.0],   "nbx": nbx, "nby": nby, "nx_gen": nx_gen, "ny_gen": ny_gen, "nx_rec": nx_rec, "ny_rec": ny_rec},
    "bin3":  {"q2_range": [4.0, 6.0],   "nbx": nbx, "nby": nby, "nx_gen": nx_gen, "ny_gen": ny_gen, "nx_rec": nx_rec, "ny_rec": ny_rec},
    "bin4":  {"q2_range": [6.0, 7.0],   "nbx": nbx, "nby": nby, "nx_gen": nx_gen, "ny_gen": ny_gen, "nx_rec": nx_rec, "ny_rec": ny_rec},
    "bin5":  {"q2_range": [7.0, 8.0],   "nbx": nbx, "nby": nby, "nx_gen": nx_gen, "ny_gen": ny_gen, "nx_rec": nx_rec, "ny_rec": ny_rec},
    "bin7":  {"q2_range": [11.0, 12.5], "nbx": nbx, "nby": nby, "nx_gen": nx_gen, "ny_gen": ny_gen, "nx_rec": nx_rec, "ny_rec": ny_rec},
    "bin9":  {"q2_range": [15.0, 17.0], "nbx": nbx, "nby": nby, "nx_gen": nx_gen, "ny_gen": ny_gen, "nx_rec": nx_rec, "ny_rec": ny_rec},
    "bin10": {"q2_range": [17.0, 23.0], "nbx": nbx, "nby": nby, "nx_gen": nx_gen, "ny_gen": ny_gen, "nx_rec": nx_rec, "ny_rec": ny_rec}
}


for binN, config in bin_configs.items():
    print(f"\n=== Procesando {binN} ===")
    nbx=config["nbx"]
    nby=config["nby"]
    nx_gen=config["nx_gen"]
    ny_gen=config["ny_gen"]
    nx_rec=config["nx_rec"]
    ny_rec=config["ny_rec"]

    eff_Gen_q2 =        select_q2_bin(eff_Gen, binN, "q2Gen")
    eff_GenFtr_q2 =     select_q2_bin(eff_GenFtr, binN, "q2Gen")
    eff_RecoGenFtr_q2 = select_q2_bin(eff_RecoGenFtr, binN, "q2Gen")
    eff_RecoFtr_q2 =    select_q2_bin(eff_RecoFtr, binN, "q2") 

    gen_x = eff_Gen_q2["gen_cosThetaL"].values  
    gen_y = eff_Gen_q2["gen_cosThetaK"].values      
    genFid_x = eff_GenFtr_q2["gen_cosThetaL"].values 
    genFid_y = eff_GenFtr_q2["gen_cosThetaK"].values
    recoFid_x = eff_RecoGenFtr_q2["CosThetaL_best"].values 
    recoFid_y = eff_RecoGenFtr_q2["CosThetaK_best"].values
    reco_x = eff_RecoFtr_q2["CosThetaL"].values
    reco_y = eff_RecoFtr_q2["CosThetaK"].values
    reco_w = eff_RecoFtr_q2["TotalWeight"].values  

    xcenters, ycenters, acc_gen, acc_gen_model, coef_acc, eff_reco, eff_reco_model, coef_reco, mask_gen = build_efficiency_2d( 
        gen_x, gen_y, genFid_x, genFid_y, recoFid_x, recoFid_y, reco_x, reco_y, weights_reco=reco_w, nbx=nbx, nby=nby, 
        nxg=nx_gen, nyg=ny_gen, nxr=nx_rec, nyr=ny_rec, min_gen=10, reg_acc=1e-5, reg_reco=1e-5 )

    # ======================================================
    # phi 
    # ======================================================
    phi_gen_all = eff_Gen_q2["gen_phi"].values
    phi_gen_fid = eff_GenFtr_q2["gen_phi"].values
    phi_reco_fid = eff_RecoGenFtr_q2["Phi_best"].values
    phi_reco = eff_RecoFtr_q2["Phi"].values
    
    centers_phi, acc_phi, acc_phi_model, coef_acc_phi, eff_reco_phi, eff_reco_phi_model, coef_reco_phi, mask_phi = build_efficiency_1d( 
        phi_gen_all, phi_gen_fid, phi_reco_fid, phi_reco, 
        weights_reco=reco_w,nbins=20, n_poly=4, reg_acc=1e-5, reg_reco=1e-5)

    plt.ioff()
    acc_model_file = f"acc_gen_model_{binN}.json"
    reco_model_file = f"eff_reco_model_{binN}.json"
    path_models = f"efficiency_models/{binN}/"
    path_plots = f"plots/projections/{binN}/"

    save_bernstein2d_model(path_models + reco_model_file, coef_reco, nx_rec, ny_rec)
    save_bernstein2d_model(path_models + acc_model_file, coef_acc, nx_gen, ny_gen)
    save_bernstein1d_model(f"efficiency_models/{binN}/acc_gen_model_phi_{binN}.json", coef_acc_phi, 4)
    save_bernstein1d_model(f"efficiency_models/{binN}/eff_reco_model_phi_{binN}.json", coef_reco_phi, 4)
    
    # ======================================================
    # PLOTs
    # ======================================================

    plot_projection_x_with_errors(xcenters, acc_gen, acc_gen_model, mask_gen, f"{binN} "+r"Gen Acceptance: $\cos\theta_\ell$", ylim=None, path=path_plots)
    plot_projection_y_with_errors( ycenters, acc_gen, acc_gen_model, mask_gen,  f"{binN} "+r"Gen Acceptance: $\cos\theta_K$",  ylim=None, path=path_plots)
    
    plot_projection_x_with_errors(xcenters, eff_reco, eff_reco_model, mask_gen, f"{binN} "+r"Reco Efficiency: $\cos\theta_\ell$", ylim=None, path=path_plots)
    plot_projection_y_with_errors(ycenters, eff_reco, eff_reco_model, mask_gen, f"{binN} "+r"Reco Efficiency: $\cos\theta_K$", ylim=None, path=path_plots)
    
    plot_projection_x_with_errors( xcenters, acc_gen*eff_reco, acc_gen_model*eff_reco_model, mask_gen, f"{binN} "+r"Total efficiency: projection cos$\theta_\ell$",ylim=None,path=path_plots)
    plot_projection_y_with_errors( xcenters, acc_gen*eff_reco, acc_gen_model*eff_reco_model, mask_gen, f"{binN} "+r"Total efficiency: projection cos$\theta_K$",ylim=None, path=path_plots)

    # Phi
    plot_1d_result(centers_phi, acc_phi, acc_phi_model, mask_phi, f"{binN} "+r"Gen Acceptance $\phi$",ylim=None,path=path_plots)
    plot_1d_result(centers_phi, eff_reco_phi, eff_reco_phi_model, mask_phi, f"{binN} "+r"Reco Efficiency $\phi$",ylim=None,path=path_plots)
    plot_1d_result(centers_phi, acc_phi*eff_reco_phi, acc_phi_model*eff_reco_phi_model, mask_phi, f"{binN} "+r"Total efficiency $\phi$",ylim=None,path=path_plots)
    plt.ion()



=== Procesando bin1 ===

=== Procesando bin2 ===

=== Procesando bin3 ===

=== Procesando bin4 ===

=== Procesando bin5 ===

=== Procesando bin7 ===

=== Procesando bin9 ===

=== Procesando bin10 ===
